# Ejercicio 3 - Workspace del IRB140 con 3 ejes

Notebook listo para Colab.

Qué hace esta versión:
- usa solo los primeros 3 ejes del IRB140;
- colorea el workspace por **destreza**;
- toma las `N` mejores semillas de destreza, no una sola;
- para cada semilla, muestrea una esfera local de 20 cm;
- para cada punto de esa esfera intenta una trayectoria lineal cartesiana;
- elige el mejor objetivo local por **menor tiempo mínimo** respetando velocidades máximas articulares;
- si hay empate, desempata con `sigma_min` más grande;
- exporta un archivo de trayectoria por cada semilla;
- muestra perfiles articulares y cartesianos para la mejor trayectoria global;
- agrega vistas 3D interactivas del workspace y de la esfera local.

## Criterio para elegir el mejor objetivo local

La **destreza** se usa solo para elegir las semillas iniciales del workspace.

Dentro de la esfera local de 20 cm, el criterio es este:
- se prueban trayectorias lineales hacia **todos** los puntos muestreados de la esfera;
- se descartan las trayectorias que no tienen IK o quedan fuera del cubo;
- para cada trayectoria factible se calcula el **tiempo mínimo** compatible con `qdot_max`;
- se elige como mejor objetivo el de **menor tiempo total mínimo**;
- si hay empate, se desempata con `sigma_min` más grande.

O sea: el mejor objetivo local **no** se elige por mayor destreza.

## Cómo leer los gráficos

- `Posición articular`: muestra cómo cambian `q1, q2, q3` en el tiempo. Sirve para ver qué hacen los ejes, pero no garantiza por sí sola que el movimiento cartesiano sea lineal.
- `Velocidad articular`: muestra qué velocidad necesita cada eje en cada tramo y cuál eje queda cerca de saturar.
- `Posición cartesiana`: muestra `x(t), y(t), z(t)`. Si la trayectoria es lineal, cada coordenada debería variar de manera suave y consistente.
- `Velocidad cartesiana`: muestra `vx(t), vy(t), vz(t)` y `|v|`. Eso ayuda a ver mejor si el avance del TCP fue aproximadamente lineal y cuánto se movió realmente.
- En el resumen impreso también aparece `desplazamiento cartesiano [m]`, que debería quedar cerca de `0.20 m` porque la esfera local es de 20 cm.

In [ ]:
# Ejecutar SOLO en Colab. Después reiniciar el runtime.
# Runtime > Restart session

%pip install roboticstoolbox-python
%pip install numpy==1.26.4 --force-reinstall
# Plotly suele venir instalado en Colab. Si hiciera falta:
# %pip install plotly

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import roboticstoolbox as rtb
import spatialmath as sm

In [ ]:
# ABB IRB140 normal, no variante T.
# 3-phase: ejes 1-3 = [200, 200, 260] deg/s
# 1-phase / IRC5 Compact: ejes 1-3 = [200, 200, 245] deg/s
QD_MAX_3_TRIFASICA = np.deg2rad(np.array([200.0, 200.0, 260.0]))
QD_MAX_3_MONOFASICA = np.deg2rad(np.array([200.0, 200.0, 245.0]))
QD_MAX_3 = QD_MAX_3_TRIFASICA.copy()

N_SEMILLAS_TOP_DESTREZA = 6
SEPARACION_MIN_SEMILLAS = 0.08
RADIO_ESFERA = 0.20
N_DIRECCIONES_ESFERA = 72
N_PASOS_TRAYECTORIA = 35
DT_NOMINAL = 0.05
EXPORT_DIRNAME = "trayectorias_generadas"

CUBO_DEFAULT = {
    "x": (-0.75, 0.75),
    "y": (-0.75, 0.75),
    "z": (0.00, 1.30),
}


@dataclass
class ResultadoTrayectoria:
    success: bool
    razon: str
    q_path: np.ndarray
    p_path: np.ndarray
    qd_nominal_max: np.ndarray
    sigma_min: float
    cumple_velocidad_nominal: bool
    dt_segmentos_min: np.ndarray
    tiempo_total_min: float
    qd_segmentos_min: np.ndarray
    qd_peak_min: np.ndarray
    desplazamiento_cartesiano: float
    longitud_cartesiana: float


@dataclass
class ResultadoSemilla:
    idx_workspace: int
    q_semilla: np.ndarray
    p_semilla: np.ndarray
    destreza: float
    sigma_min: float
    esfera: dict[str, object]
    archivo_txt_rad: str | None
    archivo_csv_deg: str | None


def crear_irb140_3ejes() -> rtb.DHRobot:
    link1 = rtb.RevoluteDH(alpha=-np.pi / 2, a=0.07, d=0.352)
    link2 = rtb.RevoluteDH(a=0.36, offset=-np.pi / 2)
    link3 = rtb.RevoluteDH(alpha=np.pi / 2, offset=np.pi)

    link4 = rtb.RevoluteDH(d=0.38, alpha=-np.pi / 2)
    link5 = rtb.RevoluteDH(alpha=np.pi / 2)
    link6 = rtb.RevoluteDH(d=0.065)
    tool = link4.A(0.0) * link5.A(0.0) * link6.A(0.0)

    robot = rtb.DHRobot([link1, link2, link3], tool=tool, name="IRB140_3EJES")
    robot.qlim = np.deg2rad(np.array([[-180.0, -100.0, -220.0], [180.0, 100.0, 60.0]]))
    return robot


def fibonacci_sphere(n: int, radio: float = 1.0) -> np.ndarray:
    indices = np.arange(n, dtype=float) + 0.5
    phi = np.arccos(1.0 - 2.0 * indices / n)
    theta = np.pi * (1.0 + np.sqrt(5.0)) * indices
    puntos = np.column_stack((np.cos(theta) * np.sin(phi), np.sin(theta) * np.sin(phi), np.cos(phi)))
    return radio * puntos


def en_cubo(p: np.ndarray, cubo: dict[str, tuple[float, float]]) -> bool:
    return cubo["x"][0] <= p[0] <= cubo["x"][1] and cubo["y"][0] <= p[1] <= cubo["y"][1] and cubo["z"][0] <= p[2] <= cubo["z"][1]


def metricas_jacobiano(robot: rtb.DHRobot, q: np.ndarray) -> dict[str, float]:
    jv = np.asarray(robot.jacob0(q)[:3, :3], dtype=float)
    singulares = np.linalg.svd(jv, compute_uv=False)
    sigma_max = float(singulares[0])
    sigma_min = float(singulares[-1])
    manipulabilidad = float(np.prod(singulares))
    destreza = sigma_min / sigma_max if sigma_max > 1e-9 else 0.0
    condicion = sigma_max / sigma_min if sigma_min > 1e-9 else np.inf
    return {"manipulabilidad": manipulabilidad, "destreza": destreza, "condicion": condicion, "sigma_min": sigma_min}


def analizar_tiempos_trayectoria(q_path: np.ndarray, qd_max: np.ndarray):
    if len(q_path) < 2:
        vacio = np.zeros((0, q_path.shape[1] if q_path.ndim == 2 else 3), dtype=float)
        return np.zeros(0), 0.0, vacio, np.zeros(3)
    dq = np.diff(q_path, axis=0)
    dt_segmentos = np.max(np.abs(dq) / qd_max[None, :], axis=1)
    dt_segmentos = np.maximum(dt_segmentos, 1e-6)
    qd_segmentos = dq / dt_segmentos[:, None]
    qd_peak = np.max(np.abs(qd_segmentos), axis=0)
    tiempo_total = float(np.sum(dt_segmentos))
    return dt_segmentos, tiempo_total, qd_segmentos, qd_peak


def analizar_geometria_cartesiana(p_path: np.ndarray):
    if len(p_path) < 2:
        return 0.0, 0.0
    desplazamiento = float(np.linalg.norm(p_path[-1] - p_path[0]))
    longitud = float(np.sum(np.linalg.norm(np.diff(p_path, axis=0), axis=1)))
    return desplazamiento, longitud


def resolver_ik_posicion(robot: rtb.DHRobot, p_objetivo: np.ndarray, q0: np.ndarray | None = None, tol: float = 1e-4, max_iter: int = 100, damping: float = 1e-3):
    q = np.zeros(robot.n) if q0 is None else np.array(q0, dtype=float).copy()
    p_objetivo = np.array(p_objetivo, dtype=float)
    try:
        sol = robot.ikine_LM(sm.SE3(*p_objetivo), q0=q, mask=[1, 1, 1, 0, 0, 0], joint_limits=True)
        if getattr(sol, "success", False):
            return True, np.asarray(sol.q, dtype=float), "ikine_LM"
    except Exception:
        pass
    q_inf = np.asarray(robot.qlim[0], dtype=float)
    q_sup = np.asarray(robot.qlim[1], dtype=float)
    for _ in range(max_iter):
        p_actual = np.asarray(robot.fkine(q).t, dtype=float)
        error = p_objetivo - p_actual
        if np.linalg.norm(error) < tol:
            return True, q, "jacobiano"
        jv = np.asarray(robot.jacob0(q)[:3, :3], dtype=float)
        sistema = jv @ jv.T + (damping**2) * np.eye(3)
        dq = jv.T @ np.linalg.solve(sistema, error)
        q = np.clip(q + dq, q_inf, q_sup)
    return False, q, "sin_convergencia"


def muestrear_workspace(robot: rtb.DHRobot, n_q: tuple[int, int, int] = (25, 21, 25), cubo: dict[str, tuple[float, float]] | None = None):
    cubo = CUBO_DEFAULT if cubo is None else cubo
    q1 = np.linspace(robot.qlim[0, 0], robot.qlim[1, 0], n_q[0])
    q2 = np.linspace(robot.qlim[0, 1], robot.qlim[1, 1], n_q[1])
    q3 = np.linspace(robot.qlim[0, 2], robot.qlim[1, 2], n_q[2])
    qs, pos, manipulabilidad, destreza, sigma_min, condicion = [], [], [], [], [], []
    for a1 in q1:
        for a2 in q2:
            for a3 in q3:
                q = np.array([a1, a2, a3], dtype=float)
                p = np.asarray(robot.fkine(q).t, dtype=float)
                if not en_cubo(p, cubo):
                    continue
                met = metricas_jacobiano(robot, q)
                qs.append(q)
                pos.append(p)
                manipulabilidad.append(met["manipulabilidad"])
                destreza.append(met["destreza"])
                sigma_min.append(met["sigma_min"])
                condicion.append(met["condicion"])
    return {"q": np.asarray(qs), "pos": np.asarray(pos), "manipulabilidad": np.asarray(manipulabilidad), "destreza": np.asarray(destreza), "sigma_min": np.asarray(sigma_min), "condicion": np.asarray(condicion)}


def voxelizar_workspace(posiciones: np.ndarray, cubo: dict[str, tuple[float, float]] | None = None, paso: float = 0.05):
    cubo = CUBO_DEFAULT if cubo is None else cubo
    origen = np.array([cubo["x"][0], cubo["y"][0], cubo["z"][0]], dtype=float)
    ext = np.array([cubo["x"][1] - cubo["x"][0], cubo["y"][1] - cubo["y"][0], cubo["z"][1] - cubo["z"][0]], dtype=float)
    dims = np.floor(ext / paso).astype(int) + 1
    indices = np.floor((posiciones - origen) / paso).astype(int)
    mascara = np.all((indices >= 0) & (indices < dims), axis=1)
    ocupados = {tuple(ind) for ind in indices[mascara]}
    total = int(np.prod(dims))
    return {"voxels_ocupados": len(ocupados), "voxels_totales": total, "fraccion_ocupada": len(ocupados) / total, "paso": paso}


def seleccionar_semillas_top_n(workspace: dict[str, np.ndarray], n: int = N_SEMILLAS_TOP_DESTREZA, criterio: str = "destreza", separacion_min_cart: float = SEPARACION_MIN_SEMILLAS):
    orden = np.argsort(workspace[criterio])[::-1]
    seleccionados = []
    for idx in orden:
        if len(seleccionados) >= n:
            break
        if not seleccionados:
            seleccionados.append(int(idx))
            continue
        p = workspace["pos"][idx]
        distancias = np.linalg.norm(workspace["pos"][seleccionados] - p, axis=1)
        if np.all(distancias >= separacion_min_cart):
            seleccionados.append(int(idx))
    if len(seleccionados) < n:
        for idx in orden:
            if len(seleccionados) >= n:
                break
            if int(idx) not in seleccionados:
                seleccionados.append(int(idx))
    return seleccionados


def evaluar_trayectoria_lineal(robot: rtb.DHRobot, q_inicio: np.ndarray, p_fin: np.ndarray, n_pasos: int = N_PASOS_TRAYECTORIA, dt_nominal: float = DT_NOMINAL, qd_max: np.ndarray | None = None):
    qd_max = QD_MAX_3 if qd_max is None else np.asarray(qd_max, dtype=float)
    q_actual = np.asarray(q_inicio, dtype=float).copy()
    p_inicio = np.asarray(robot.fkine(q_actual).t, dtype=float)
    puntos = np.linspace(p_inicio, p_fin, n_pasos)
    q_path = [q_actual.copy()]
    p_path = [p_inicio.copy()]
    qd_nominal_max = np.zeros(robot.n, dtype=float)
    sigma_min = math.inf
    for p_obj in puntos[1:]:
        ok, q_nuevo, metodo = resolver_ik_posicion(robot, p_obj, q0=q_actual)
        if not ok:
            return ResultadoTrayectoria(False, f"ik_fallida_{metodo}", np.asarray(q_path), np.asarray(p_path), qd_nominal_max, 0.0, False, np.zeros(0), 0.0, np.zeros((0, robot.n)), np.zeros(robot.n), 0.0, 0.0)
        dq = q_nuevo - q_actual
        qd = np.abs(dq / dt_nominal)
        qd_nominal_max = np.maximum(qd_nominal_max, qd)
        met = metricas_jacobiano(robot, q_nuevo)
        sigma_min = min(sigma_min, met["sigma_min"])
        q_actual = q_nuevo
        q_path.append(q_actual.copy())
        p_path.append(np.asarray(robot.fkine(q_actual).t, dtype=float))
    q_path = np.asarray(q_path)
    p_path = np.asarray(p_path)
    dt_segmentos_min, tiempo_total_min, qd_segmentos_min, qd_peak_min = analizar_tiempos_trayectoria(q_path, qd_max)
    desplazamiento_cartesiano, longitud_cartesiana = analizar_geometria_cartesiana(p_path)
    cumple_velocidad_nominal = bool(np.all(qd_nominal_max <= qd_max))
    return ResultadoTrayectoria(True, "ok", q_path, p_path, qd_nominal_max, float(sigma_min), cumple_velocidad_nominal, dt_segmentos_min, tiempo_total_min, qd_segmentos_min, qd_peak_min, desplazamiento_cartesiano, longitud_cartesiana)


def muestrear_esfera_local(robot: rtb.DHRobot, q_centro: np.ndarray, radio: float = RADIO_ESFERA, n_direcciones: int = N_DIRECCIONES_ESFERA, n_pasos: int = N_PASOS_TRAYECTORIA, dt_nominal: float = DT_NOMINAL, cubo: dict[str, tuple[float, float]] | None = None, qd_max: np.ndarray | None = None):
    cubo = CUBO_DEFAULT if cubo is None else cubo
    qd_max = QD_MAX_3 if qd_max is None else np.asarray(qd_max, dtype=float)
    p_centro = np.asarray(robot.fkine(q_centro).t, dtype=float)
    direcciones = fibonacci_sphere(n_direcciones, radio=radio)
    endpoints, factibles, factibles_nominal, sigmas = [], [], [], []
    vel_nominal, tiempos_min, qd_peak_min, resultados = [], [], [], []
    for d in direcciones:
        p_fin = p_centro + d
        if not en_cubo(p_fin, cubo):
            endpoints.append(p_fin)
            factibles.append(False)
            factibles_nominal.append(False)
            sigmas.append(0.0)
            vel_nominal.append(np.zeros(robot.n))
            tiempos_min.append(np.inf)
            qd_peak_min.append(np.zeros(robot.n))
            resultados.append(ResultadoTrayectoria(False, "fuera_del_cubo", np.zeros((0, robot.n)), np.zeros((0, 3)), np.zeros(robot.n), 0.0, False, np.zeros(0), np.inf, np.zeros((0, robot.n)), np.zeros(robot.n), 0.0, 0.0))
            continue
        res = evaluar_trayectoria_lineal(robot, q_centro, p_fin, n_pasos=n_pasos, dt_nominal=dt_nominal, qd_max=qd_max)
        endpoints.append(p_fin)
        factibles.append(res.success)
        factibles_nominal.append(res.success and res.cumple_velocidad_nominal)
        sigmas.append(res.sigma_min)
        vel_nominal.append(res.qd_nominal_max)
        tiempos_min.append(res.tiempo_total_min)
        qd_peak_min.append(res.qd_peak_min)
        resultados.append(res)
    factibles = np.asarray(factibles, dtype=bool)
    factibles_nominal = np.asarray(factibles_nominal, dtype=bool)
    endpoints = np.asarray(endpoints, dtype=float)
    sigmas = np.asarray(sigmas, dtype=float)
    vel_nominal = np.asarray(vel_nominal, dtype=float)
    tiempos_min = np.asarray(tiempos_min, dtype=float)
    qd_peak_min = np.asarray(qd_peak_min, dtype=float)
    mejor_idx = None
    if np.any(factibles):
        idxs = np.where(factibles)[0]
        orden = np.lexsort((-sigmas[idxs], tiempos_min[idxs]))
        mejor_idx = int(idxs[orden[0]])
    return {"centro_q": np.asarray(q_centro), "centro_p": p_centro, "endpoints": endpoints, "factibles": factibles, "factibles_nominal": factibles_nominal, "sigmas": sigmas, "vel_nominal": vel_nominal, "tiempos_min": tiempos_min, "qd_peak_min": qd_peak_min, "resultados": resultados, "mejor_idx": mejor_idx, "mejor_resultado": None if mejor_idx is None else resultados[mejor_idx], "fraccion_factible": float(np.mean(factibles)), "fraccion_factible_nominal": float(np.mean(factibles_nominal))}


def qpath3_a_qpath6(q_path3: np.ndarray) -> np.ndarray:
    q_path6 = np.zeros((len(q_path3), 6), dtype=float)
    q_path6[:, :3] = q_path3
    return q_path6


def exportar_trayectoria(resultado: ResultadoTrayectoria, nombre_base: str, salida_dir: Path | None = None):
    if not resultado.success or len(resultado.q_path) < 2:
        return None, None
    salida_dir = Path.cwd() / EXPORT_DIRNAME if salida_dir is None else Path(salida_dir)
    salida_dir.mkdir(parents=True, exist_ok=True)
    tiempos = np.concatenate(([0.0], np.cumsum(resultado.dt_segmentos_min)))
    q6_rad = qpath3_a_qpath6(resultado.q_path)
    q6_deg = np.rad2deg(q6_rad)
    data_rad = np.column_stack((tiempos, q6_rad))
    data_deg = np.column_stack((tiempos, q6_deg))
    txt_rad = salida_dir / f"{nombre_base}_rad.txt"
    csv_deg = salida_dir / f"{nombre_base}_deg.csv"
    np.savetxt(txt_rad, data_rad, fmt="%.6f")
    np.savetxt(csv_deg, data_deg, fmt="%.6f", delimiter=",", header="t,q1,q2,q3,q4,q5,q6", comments="")
    return str(txt_rad), str(csv_deg)


def evaluar_semillas_top_n(robot: rtb.DHRobot, workspace: dict[str, np.ndarray], idx_semillas: list[int], radio: float = RADIO_ESFERA, n_direcciones: int = N_DIRECCIONES_ESFERA, n_pasos: int = N_PASOS_TRAYECTORIA, dt_nominal: float = DT_NOMINAL, cubo: dict[str, tuple[float, float]] | None = None, qd_max: np.ndarray | None = None):
    resultados = []
    for rank, idx in enumerate(idx_semillas, start=1):
        q_semilla = workspace["q"][idx]
        p_semilla = workspace["pos"][idx]
        esfera = muestrear_esfera_local(robot, q_centro=q_semilla, radio=radio, n_direcciones=n_direcciones, n_pasos=n_pasos, dt_nominal=dt_nominal, cubo=cubo, qd_max=qd_max)
        nombre_base = f"trayectoria_semilla_{rank:02d}"
        archivo_txt_rad, archivo_csv_deg = exportar_trayectoria(esfera["mejor_resultado"], nombre_base)
        resultados.append(ResultadoSemilla(int(idx), np.asarray(q_semilla), np.asarray(p_semilla), float(workspace["destreza"][idx]), float(workspace["sigma_min"][idx]), esfera, archivo_txt_rad, archivo_csv_deg))
    return resultados


def elegir_mejor_semilla_global(resultados_semillas: list[ResultadoSemilla]):
    candidatos = [r for r in resultados_semillas if r.esfera["mejor_resultado"] is not None]
    if not candidatos:
        return None
    return min(candidatos, key=lambda r: (r.esfera["mejor_resultado"].tiempo_total_min, -r.esfera["mejor_resultado"].sigma_min))


def graficar_workspace(workspace: dict[str, np.ndarray], semilla_idx: int | None = None, semillas_idx: list[int] | None = None, metrica: str = "destreza"):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    scatter = ax.scatter(workspace["pos"][:, 0], workspace["pos"][:, 1], workspace["pos"][:, 2], c=workspace[metrica], s=10, cmap="viridis", alpha=0.65)
    fig.colorbar(scatter, ax=ax, label=metrica)
    if semillas_idx is not None and len(semillas_idx) > 0:
        ps = workspace["pos"][semillas_idx]
        ax.scatter(ps[:, 0], ps[:, 1], ps[:, 2], color="red", s=60, label="top semillas")
        ax.legend()
    elif semilla_idx is not None:
        p = workspace["pos"][semilla_idx]
        ax.scatter(p[0], p[1], p[2], color="red", s=90, label="semilla")
        ax.legend()
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title("Workspace 3R del IRB140")
    plt.show()


def graficar_workspace_interactivo(workspace: dict[str, np.ndarray], semilla_idx: int | None = None, semillas_idx: list[int] | None = None, metrica: str = "destreza"):
    import plotly.graph_objects as go
    fig = go.Figure(data=[go.Scatter3d(x=workspace["pos"][:, 0], y=workspace["pos"][:, 1], z=workspace["pos"][:, 2], mode="markers", marker=dict(size=3, color=workspace[metrica], colorscale="Viridis", opacity=0.7, colorbar=dict(title=metrica)), name="workspace")])
    if semillas_idx is not None and len(semillas_idx) > 0:
        ps = workspace["pos"][semillas_idx]
        fig.add_trace(go.Scatter3d(x=ps[:, 0], y=ps[:, 1], z=ps[:, 2], mode="markers", marker=dict(size=6, color="red"), name="top semillas"))
    elif semilla_idx is not None:
        p = workspace["pos"][semilla_idx]
        fig.add_trace(go.Scatter3d(x=[p[0]], y=[p[1]], z=[p[2]], mode="markers", marker=dict(size=7, color="red"), name="semilla"))
    fig.update_layout(title="Workspace 3R del IRB140", scene=dict(xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]", aspectmode="data"), margin=dict(l=0, r=0, b=0, t=40))
    fig.show()


def graficar_esfera_local(esfera):
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")
    endpoints = np.asarray(esfera["endpoints"])
    factibles = np.asarray(esfera["factibles"])
    centro = np.asarray(esfera["centro_p"])
    mejor_idx = esfera["mejor_idx"]
    ax.scatter(endpoints[~factibles, 0], endpoints[~factibles, 1], endpoints[~factibles, 2], color="lightgray", s=20, label="no factible")
    ax.scatter(endpoints[factibles, 0], endpoints[factibles, 1], endpoints[factibles, 2], color="tab:green", s=30, label="factible")
    ax.scatter(centro[0], centro[1], centro[2], color="red", s=90, label="centro")
    if mejor_idx is not None:
        mejor = endpoints[mejor_idx]
        camino = esfera["mejor_resultado"].p_path
        ax.scatter(mejor[0], mejor[1], mejor[2], color="orange", s=70, label="mejor objetivo")
        ax.plot(camino[:, 0], camino[:, 1], camino[:, 2], color="orange", linewidth=2)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title("Esfera local de 20 cm")
    ax.legend()
    plt.show()


def graficar_esfera_local_interactiva(esfera, titulo: str = "Esfera local de 20 cm"):
    import plotly.graph_objects as go
    endpoints = np.asarray(esfera["endpoints"])
    factibles = np.asarray(esfera["factibles"])
    centro = np.asarray(esfera["centro_p"])
    mejor_idx = esfera["mejor_idx"]
    fig = go.Figure()
    if np.any(~factibles):
        fig.add_trace(go.Scatter3d(x=endpoints[~factibles, 0], y=endpoints[~factibles, 1], z=endpoints[~factibles, 2], mode="markers", marker=dict(size=3, color="lightgray"), name="no factible"))
    if np.any(factibles):
        fig.add_trace(go.Scatter3d(x=endpoints[factibles, 0], y=endpoints[factibles, 1], z=endpoints[factibles, 2], mode="markers", marker=dict(size=4, color="green"), name="factible"))
    fig.add_trace(go.Scatter3d(x=[centro[0]], y=[centro[1]], z=[centro[2]], mode="markers", marker=dict(size=7, color="red"), name="centro"))
    if mejor_idx is not None:
        mejor = endpoints[mejor_idx]
        camino = esfera["mejor_resultado"].p_path
        fig.add_trace(go.Scatter3d(x=[mejor[0]], y=[mejor[1]], z=[mejor[2]], mode="markers", marker=dict(size=7, color="orange"), name="mejor objetivo"))
        fig.add_trace(go.Scatter3d(x=camino[:, 0], y=camino[:, 1], z=camino[:, 2], mode="lines", line=dict(color="orange", width=6), name="trayectoria"))
    fig.update_layout(title=titulo, scene=dict(xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]", aspectmode="data"), margin=dict(l=0, r=0, b=0, t=40))
    fig.show()


def graficar_perfiles_trayectoria(resultado: ResultadoTrayectoria, qd_max: np.ndarray | None = None):
    qd_max = QD_MAX_3 if qd_max is None else np.asarray(qd_max, dtype=float)
    if not resultado.success or len(resultado.q_path) < 2:
        raise ValueError("No hay una trayectoria valida para graficar.")
    tiempos = np.concatenate(([0.0], np.cumsum(resultado.dt_segmentos_min)))
    q_deg = np.rad2deg(resultado.q_path)
    qd_deg = np.rad2deg(resultado.qd_segmentos_min)
    qd_max_deg = np.rad2deg(qd_max)
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=False)
    for i in range(q_deg.shape[1]):
        axes[0].plot(tiempos, q_deg[:, i], marker="o", label=f"q{i+1}")
    axes[0].set_title("Posicion articular de la mejor trayectoria")
    axes[0].set_xlabel("t [s]")
    axes[0].set_ylabel("q [deg]")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    t_mid = 0.5 * (tiempos[:-1] + tiempos[1:])
    for i in range(qd_deg.shape[1]):
        axes[1].step(t_mid, np.abs(qd_deg[:, i]), where="mid", label=f"|q{i+1}_dot|")
        axes[1].axhline(qd_max_deg[i], linestyle="--", linewidth=1, color=axes[1].lines[-1].get_color())
    axes[1].set_title("Velocidad articular minima requerida por tramo")
    axes[1].set_xlabel("t [s]")
    axes[1].set_ylabel("|q_dot| [deg/s]")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    fig.tight_layout()
    plt.show()


def graficar_perfiles_cartesianos(resultado: ResultadoTrayectoria):
    if not resultado.success or len(resultado.p_path) < 2:
        raise ValueError("No hay una trayectoria valida para graficar.")
    tiempos = np.concatenate(([0.0], np.cumsum(resultado.dt_segmentos_min)))
    p = resultado.p_path
    v_xyz = np.diff(p, axis=0) / resultado.dt_segmentos_min[:, None]
    v_norma = np.linalg.norm(v_xyz, axis=1)
    t_mid = 0.5 * (tiempos[:-1] + tiempos[1:])
    etiquetas = ["x", "y", "z"]
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=False)
    for i, etq in enumerate(etiquetas):
        axes[0].plot(tiempos, p[:, i], marker="o", label=etq)
    axes[0].set_title("Posicion cartesiana de la mejor trayectoria")
    axes[0].set_xlabel("t [s]")
    axes[0].set_ylabel("p [m]")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    for i, etq in enumerate(etiquetas):
        axes[1].step(t_mid, v_xyz[:, i], where="mid", label=f"v{etq}")
    axes[1].step(t_mid, v_norma, where="mid", linestyle="--", color="black", label="|v|")
    axes[1].set_title("Velocidad cartesiana por tramo")
    axes[1].set_xlabel("t [s]")
    axes[1].set_ylabel("v [m/s]")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    fig.tight_layout()
    plt.show()


def imprimir_resumen_semillas(workspace: dict[str, np.ndarray], voxel: dict[str, float | int], resultados_semillas: list[ResultadoSemilla], mejor_global: ResultadoSemilla | None):
    print("=== RESUMEN WORKSPACE 3R ===")
    print(f"Puntos cartesianos dentro del cubo: {len(workspace['pos'])}")
    print(f"Fraccion de voxels ocupados: {voxel['fraccion_ocupada']:.3f}")
    print(f"Voxelizacion usada: {voxel['paso']:.3f} m")
    print()
    print("=== TOP SEMILLAS POR DESTREZA ===")
    for rank, res_semilla in enumerate(resultados_semillas, start=1):
        esfera = res_semilla.esfera
        mejor = esfera["mejor_resultado"]
        print(f"Semilla {rank}: idx={res_semilla.idx_workspace}, destreza={res_semilla.destreza:.4f}, sigma_min={res_semilla.sigma_min:.4f}")
        print("  q [deg] =", np.round(np.rad2deg(res_semilla.q_semilla), 2))
        print("  p [m]   =", np.round(res_semilla.p_semilla, 4))
        print(f"  fraccion factible = {esfera['fraccion_factible']:.3f}")
        if mejor is not None:
            print("  mejor p_obj [m] =", np.round(esfera["endpoints"][esfera["mejor_idx"]], 4))
            print(f"  desplazamiento [m] = {mejor.desplazamiento_cartesiano:.4f}")
            print(f"  longitud camino [m] = {mejor.longitud_cartesiana:.4f}")
            print(f"  tiempo minimo [s] = {mejor.tiempo_total_min:.4f}")
            print("  qd pico minimo [deg/s] =", np.round(np.rad2deg(mejor.qd_peak_min), 2))
            print("  archivo rad =", res_semilla.archivo_txt_rad)
        else:
            print("  sin trayectoria factible")
        print()
    if mejor_global is not None:
        mejor = mejor_global.esfera["mejor_resultado"]
        print("=== MEJOR SEMILLA GLOBAL ===")
        print("idx workspace =", mejor_global.idx_workspace)
        print("q_semilla [deg] =", np.round(np.rad2deg(mejor_global.q_semilla), 2))
        print("p_semilla [m] =", np.round(mejor_global.p_semilla, 4))
        print("mejor objetivo [m] =", np.round(mejor_global.esfera["endpoints"][mejor_global.esfera["mejor_idx"]], 4))
        print(f"desplazamiento cartesiano [m] = {mejor.desplazamiento_cartesiano:.4f}")
        print(f"longitud del camino [m] = {mejor.longitud_cartesiana:.4f}")
        print(f"tiempo total minimo [s] = {mejor.tiempo_total_min:.4f}")
        print("qd pico minimo [deg/s] =", np.round(np.rad2deg(mejor.qd_peak_min), 2))
        print("archivo txt rad =", mejor_global.archivo_txt_rad)
        print("archivo csv deg =", mejor_global.archivo_csv_deg)

In [ ]:
robot = crear_irb140_3ejes()
print(robot)

# Si tarda mucho en Colab, bajar estos parametros.
workspace = muestrear_workspace(robot, n_q=(27, 23, 27), cubo=CUBO_DEFAULT)
voxel = voxelizar_workspace(workspace["pos"], cubo=CUBO_DEFAULT, paso=0.05)

idx_semillas = seleccionar_semillas_top_n(
    workspace,
    n=N_SEMILLAS_TOP_DESTREZA,
    criterio="destreza",
    separacion_min_cart=SEPARACION_MIN_SEMILLAS,
)

resultados_semillas = evaluar_semillas_top_n(
    robot,
    workspace,
    idx_semillas,
    radio=RADIO_ESFERA,
    n_direcciones=N_DIRECCIONES_ESFERA,
    n_pasos=N_PASOS_TRAYECTORIA,
    dt_nominal=DT_NOMINAL,
    cubo=CUBO_DEFAULT,
    qd_max=QD_MAX_3,
)

mejor_global = elegir_mejor_semilla_global(resultados_semillas)
imprimir_resumen_semillas(workspace, voxel, resultados_semillas, mejor_global)

In [ ]:
graficar_workspace(workspace, semillas_idx=idx_semillas, metrica="destreza")
if mejor_global is not None:
    graficar_esfera_local(mejor_global.esfera)
    graficar_perfiles_trayectoria(mejor_global.esfera["mejor_resultado"], qd_max=QD_MAX_3)
    graficar_perfiles_cartesianos(mejor_global.esfera["mejor_resultado"])

In [ ]:
graficar_workspace_interactivo(workspace, semillas_idx=idx_semillas, metrica="destreza")
if mejor_global is not None:
    graficar_esfera_local_interactiva(mejor_global.esfera, titulo="Esfera local de la mejor semilla global")

In [ ]:
for i, res in enumerate(resultados_semillas, start=1):
    print(f"Semilla {i}: txt_rad={res.archivo_txt_rad} | csv_deg={res.archivo_csv_deg}")